# Granular speech codec: training and inspection

This notebook trains a small speech codec and then opens it up, step by step,
to show what each part does.

**The codec in one line:** speech `S` → encoder → latent `z` (50 frames per second)
→ quantizer (sends one code index per group per frame) → decoder → speech `Ŝ`.

**The question:** does a *fuzzy* quantizer (fuzzy c-means granulation and
degranulation) code speech as well as, or better than, standard VQ and scalar
quantization at the same bit rate?

Part 1 sets things up and trains. Part 2 inspects a trained run.

## Part 1. Setup

**Cell 1** copies the code from the read-only input folder to `/kaggle/working`,
where files can be written.

In [ ]:
import glob, os, shutil
DATA = "/kaggle/input/datasets/matinazad/granular-speech-codec"
CODE = "/kaggle/working/granular-speech-codec"
src = os.path.dirname(glob.glob(DATA + "/**/requirements.txt", recursive=True)[0])
shutil.rmtree(CODE, ignore_errors=True)
shutil.copytree(src, CODE)
print(sorted(os.listdir(CODE)))

**Cell 2** installs the packages and runs the checks. Expect `15 passed`.
The last test loads `transformers` and may pause for a minute.

In [ ]:
%cd /kaggle/working/granular-speech-codec
!pip install -q -r requirements.txt
!python -m pytest -q tests

**Cell 3** downloads LibriSpeech (read English audiobooks, 16 kHz):
`train-clean-100` (about 100 hours) for training and `dev-clean` for evaluation.
Each archive is deleted after unpacking to save disk space.

In [ ]:
%cd /kaggle/working
!wget -q https://www.openslr.org/resources/12/dev-clean.tar.gz && tar xzf dev-clean.tar.gz && rm dev-clean.tar.gz
!wget -q https://www.openslr.org/resources/12/train-clean-100.tar.gz && tar xzf train-clean-100.tar.gz && rm train-clean-100.tar.gz
!ls LibriSpeech

**Cell 4: training.** The loss has five terms:

| Term | What it asks for |
|---|---|
| `stft` | the decoded spectrum matches the original at three time-frequency resolutions (phase ignored) |
| `env` | the loudness profile (smoothed log-energy, in dB) matches |
| `aux` | the quantizer's own loss; for the fuzzy quantizer this is the c-means objective `J = Σ u_i^m ‖x − v_i‖²` |
| `content` | the speech recognizer gives the same character probabilities per frame (KL divergence) |
| `speaker` | the speaker-identity model gives the same speaker embedding (cosine distance) |

The fuzzy quantizer has two schedule values printed in the log:
`m` (fuzzifier, 1.5 → 1.05) and `alpha` (0 = soft output, 1 = hard prototype).

Start with 200 steps to measure the speed (`sec` column), then raise `--steps`.

In [ ]:
%cd /kaggle/working/granular-speech-codec
!python train.py --data_dir /kaggle/working/LibriSpeech/train-clean-100 \
    --eval_dir /kaggle/working/LibriSpeech/dev-clean \
    --quantizer fuzzy --steps 200 --eval_every 200 --eval_items 40 \
    --w_content 1 --w_speaker 1 --out runs/timing

## Part 2. Inspect a trained run

Set `RUN` to the run folder you want to look at. Everything below reads the
checkpoint and the logs from that folder. With only 200 training steps the
codec is barely trained, so expect noisy audio; the point is to see the
machinery. Run again after a long training run to see the real behavior.

In [ ]:
import json, os
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import Audio, HTML, display

os.chdir("/kaggle/working/granular-speech-codec")
from gsc.model import Codec
from gsc.data import AudioFolder
from gsc.quantizers import _split_groups
from gsc.rate import IndexHistogram

RUN = "runs/timing"
EVAL_DIR = "/kaggle/working/LibriSpeech/dev-clean"
N_CLIPS = 8           # 4-second clips used below
SR = 16000
device = "cuda" if torch.cuda.is_available() else "cpu"

ck = torch.load(f"{RUN}/checkpoint.pt", map_location=device, weights_only=False)
A = ck["args"]
fsq_levels = [int(v) for v in A["fsq_levels"].split(",")] if A.get("fsq_levels") else None
model = Codec(A["quantizer"], A["channels"], latent_dim=A["latent_dim"], groups=A["groups"],
              codebook_size=A["codebook_size"], code_dim=A["code_dim"], fsq_levels=fsq_levels).to(device)
model.load_state_dict(ck["model"])
model.eval()
QNAME, G, K = A["quantizer"], A["groups"], model.codebook_size
M_END = A["m_end"]

ds = AudioFolder(EVAL_DIR, "all", segment=4 * SR, max_files=N_CLIPS)
X = torch.stack([ds[i] for i in range(len(ds))]).to(device)       # (N, 1, 64000)
with torch.no_grad():
    Y, Q = model(X, m=M_END, alpha=1.0)                            # decoded speech
    P = _split_groups(model.quantizer.proj_in(Q["z"]), G)          # (N, G, T, d) pre-quantization points
FPS = SR / model.hop
print(f"run {RUN}: quantizer={QNAME}, step {ck['step']}, groups={G}, K={K}, "
      f"{FPS:.0f} frames/s, nominal {model.nominal_bits_per_frame() * FPS:.0f} bit/s")
print("latent points per group:", tuple(P.shape))

### 2.1 Learning curves

Each training term on its own axis, so a small term is not hidden by a big
one. The last panel shows the fuzzy schedule: `m` falls toward 1 and `alpha`
rises to 1 over the first half of training. Watch for a jump in the losses
when `alpha` reaches 1: that is the moment the decoder starts receiving hard
prototypes instead of soft mixtures.

In [ ]:
log = pd.read_json(f"{RUN}/train_log.jsonl", lines=True)
terms = [c for c in ["loss", "stft", "env", "aux", "content", "speaker"] if c in log]
fig, axes = plt.subplots(1, len(terms) + 1, figsize=(3.2 * (len(terms) + 1), 3))
for ax, c in zip(axes, terms):
    ax.plot(log["step"], log[c], ".-")
    ax.set_title(c); ax.set_xlabel("step"); ax.grid(alpha=0.3)
ax = axes[-1]
ax.plot(log["step"], log["m"], label="m (fuzzifier)")
ax.plot(log["step"], log["alpha"], label="alpha (hardness)")
ax.set_title("fuzzy schedule"); ax.set_xlabel("step"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

ev = pd.read_json(f"{RUN}/eval.jsonl", lines=True)
display(ev.drop(columns=["codes_used"]).round(4))

### 2.2 Listen: original vs decoded

`normalize=False` keeps the true loudness, so a quiet output sounds quiet.
Below each pair: the two spectrograms (time on x, frequency on y, color =
log magnitude in dB). Missing high-frequency energy or smeared harmonics show
up here before you hear them.

In [ ]:
def spec_db(x):
    s = torch.stft(torch.as_tensor(x), 512, 128, window=torch.hann_window(512), return_complex=True)
    return 20 * torch.log10(s.abs() + 1e-6).numpy()

for i in range(min(3, len(X))):
    x, y = X[i, 0].cpu().numpy(), Y[i, 0].cpu().numpy()
    display(HTML(f"<b>clip {i}</b>: original, then decoded"))
    display(Audio(x, rate=SR, normalize=False))
    display(Audio(np.clip(y, -1, 1), rate=SR, normalize=False))
    fig, axes = plt.subplots(1, 2, figsize=(12, 2.6), sharey=True)
    for ax, s, t in [(axes[0], x, "original"), (axes[1], y, "decoded")]:
        ax.imshow(spec_db(s), origin="lower", aspect="auto", vmin=-80, vmax=20,
                  extent=[0, len(s) / SR, 0, SR / 2000])
        ax.set_title(t); ax.set_xlabel("time (s)")
    axes[0].set_ylabel("kHz"); plt.tight_layout(); plt.show()

### 2.3 Bit rate and code usage

Each group sends one index from 0 to K−1 per frame. If all K codes were used
equally, each index would cost log₂K bits. In practice some codes are used
far more than others, and an entropy coder spends fewer bits on frequent codes.
The entropy

  H = −Σ_k p_k log₂ p_k  (bits per index)

is that lower cost. Rate = Σ over groups of H × 50 frames/s.

The bars show how often each code was used, sorted. A short tail means only a
few codes are alive (codebook collapse); a long flat tail means the codebook
is well used.

In [ ]:
hist = IndexHistogram(G, K)
hist.update(Q["indices"])
p = hist.counts / hist.counts.sum(1, keepdim=True)
H = -(p * torch.log2(p.clamp_min(1e-300))).sum(1)
fig, axes = plt.subplots(1, G, figsize=(3.2 * G, 2.6), sharey=True)
for g in range(G):
    c = np.sort(hist.counts[g].numpy())[::-1]
    axes[g].bar(np.arange(K), c, width=1.0)
    axes[g].set_yscale("log"); axes[g].set_xlim(-1, max(20, int((c > 0).sum()) + 5))
    axes[g].set_title(f"group {g}: {int((c > 0).sum())}/{K} used\nH = {H[g]:.2f} bits")
    axes[g].set_xlabel("code rank")
axes[0].set_ylabel("count")
plt.tight_layout(); plt.show()
print(f"entropy rate {H.sum().item() * FPS:.0f} bit/s  (nominal {model.nominal_bits_per_frame() * FPS:.0f} bit/s)")

### 2.4 The latent space and the prototypes (granules)

Each frame of speech becomes a point in a small space (dimension `code_dim`)
for each group. The prototypes `v_1 … v_K` are the granule centers. Quantization
replaces each point by one prototype.

The space has more than 2 dimensions, so the plot uses PCA: it projects
everything onto the 2 directions along which the points vary most (the top
two right singular vectors of the centered data matrix). Colors = assigned
code. Black ×: prototypes, sized by how often they are used in these clips.

What to look for: prototypes should sit inside the cloud of points. Prototypes
far from every point are dead codes; one prototype covering a huge region
means coarse granulation there.

In [ ]:
g = 0
pts = P[:, g].reshape(-1, P.shape[-1]).cpu().numpy()
idx = Q["indices"][:, g].reshape(-1).cpu().numpy()
mu = pts.mean(0)
_, _, Vt = np.linalg.svd(pts - mu, full_matrices=False)
proj = lambda a: (a - mu) @ Vt[:2].T
pp = proj(pts)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(pp[:, 0], pp[:, 1], c=idx % 20, cmap="tab20", s=4, alpha=0.5)
if QNAME in ("vq", "fuzzy"):
    cb = model.quantizer.codebook[g].detach().cpu().numpy()
    used = np.bincount(idx, minlength=K)
    pc = proj(cb)
    ax.scatter(pc[:, 0], pc[:, 1], marker="x", c="k", s=10 + 200 * used / used.max(), lw=1)
    print(f"group {g}: {np.sum(used > 0)} of {K} prototypes used in these clips")
else:
    print("FSQ has no learned prototypes: its codes are the fixed grid of rounded levels")
ax.set_title(f"group {g}: frames (dots) and prototypes (x), PCA view")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); plt.show()

### 2.5 Granulation over time (fuzzy quantizer only)

For one clip, group 0: the top panel is the spectrogram, the middle panel the
membership degrees `u_i(t)` of the most active prototypes, and the bottom
panel the largest membership per frame.

The membership formula (fuzzy c-means), with `d_i = ‖x − v_i‖`:

  u_i = 1 / Σ_j (d_i / d_j)^(2/(m−1))

So the memberships sum to 1, and the nearest prototype gets the largest
degree. A max membership near 1 means the frame sits clearly in one granule;
near 1/K means it sits between many. Look at whether granule changes line up
with changes in the spectrogram (phone boundaries, voiced vs silence).

In [ ]:
if QNAME != "fuzzy":
    print("This cell needs a fuzzy run.")
else:
    clip, g, top = 0, 0, 16
    u = Q["memberships"][clip, g].cpu().numpy()            # (T, K)
    order = np.argsort(-u.sum(0))[:top]
    T = u.shape[0]
    fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True,
                             gridspec_kw={"height_ratios": [2, 2, 1]})
    axes[0].imshow(spec_db(X[clip, 0].cpu().numpy()), origin="lower", aspect="auto",
                   vmin=-80, vmax=20, extent=[0, T / FPS, 0, SR / 2000])
    axes[0].set_ylabel("kHz"); axes[0].set_title(f"clip {clip}: spectrogram")
    im = axes[1].imshow(u[:, order].T, aspect="auto", origin="lower", cmap="magma",
                        extent=[0, T / FPS, -0.5, top - 0.5], vmin=0, vmax=1)
    axes[1].set_yticks(range(top)); axes[1].set_yticklabels(order, fontsize=7)
    axes[1].set_ylabel("prototype"); axes[1].set_title(f"memberships u_i(t), group {g}, m = {M_END}")
    axes[2].plot(np.arange(T) / FPS, u.max(1))
    axes[2].set_ylim(0, 1.05); axes[2].set_ylabel("max u"); axes[2].set_xlabel("time (s)")
    plt.tight_layout(); plt.show()

### 2.6 What the fuzzifier m does

The same frame, coded with different values of `m`. For each `m`:

* **left:** the 10 largest memberships of one frame. Large `m` spreads the
  membership over many prototypes; as `m → 1` it concentrates on the nearest one
  (exponent `2/(m−1)` → ∞, so the smallest distance wins).
* **right:** the average squared error between a latent point `x` and its
  reconstruction, over all frames:
  * soft (degranulation): `x̂ = Σ u_i^m v_i / Σ u_i^m`
  * hard (nearest prototype): `x̂ = v_nearest`

At large `m` the soft mixture is pulled toward the codebook mean, so its error
is large. As `m → 1` it equals the hard prototype. This is why training lowers
`m`: the decoder must work with what is actually sent, the hard prototype.

In [ ]:
if QNAME != "fuzzy":
    print("This cell needs a fuzzy run.")
else:
    g, ms = 0, [2.0, 1.5, 1.2, 1.1, 1.05]
    fq = model.quantizer
    with torch.no_grad():
        D = fq._sq_dist(P)[:, g]                       # (N, T, K) squared distances
        x = P[:, g]
        cb = fq.codebook[g]
        frame = D[0, D.shape[1] // 2]                  # one frame, middle of clip 0
        soft_err, hard_err = [], (D.min(-1).values).mean().item()
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
        for mval in ms:
            u1 = fq.memberships(frame, mval).cpu().numpy()
            axes[0].plot(np.sort(u1)[::-1][:10], "o-", label=f"m = {mval}")
            u = fq.memberships(D, mval)
            w = u ** mval; w = w / w.sum(-1, keepdim=True)
            soft = w @ cb
            soft_err.append(((x - soft) ** 2).sum(-1).mean().item())
    axes[0].set_xlabel("rank"); axes[0].set_ylabel("membership"); axes[0].legend(fontsize=8)
    axes[0].set_title("one frame: largest memberships")
    axes[1].plot(ms, soft_err, "o-", label="soft (degranulation)")
    axes[1].axhline(hard_err, color="k", ls="--", label="hard (nearest prototype)")
    axes[1].set_xlabel("m"); axes[1].set_ylabel("mean ‖x − x̂‖²"); axes[1].invert_xaxis()
    axes[1].legend(fontsize=8); axes[1].set_title("latent reconstruction error")
    plt.tight_layout(); plt.show()

### 2.7 Perceptual priors: what the codec is asked to keep

**Content prior.** A pretrained speech recognizer (wav2vec 2.0) outputs, every
20 ms, a probability vector over 32 characters (letters, apostrophe, word
separator `|`, and a blank). This vector is a membership vector over
characters. The heatmaps show it for the original and the decoded speech; the
curve shows the per-frame KL divergence

  KL(p‖q) = Σ_c p_c log(p_c / q_c)

which is zero where the decoded speech gives the same character
probabilities. Training minimizes its average.

**Speaker prior.** A speaker model (WavLM x-vector) maps each clip to a unit
vector; the similarity is the cosine (1 = same voice).

**Transcripts.** Greedy decoding of both. The word error rate uses the
original's transcript as the reference, so it measures what the codec
destroyed, not mistakes of the recognizer.

This cell downloads the two models the first time (Internet must be on).

In [ ]:
try:
    from gsc.priors import ContentPrior, SpeakerPrior, word_error_rate
    content, speaker = ContentPrior().to(device), SpeakerPrior().to(device)
except Exception as e:
    content = speaker = None
    print("priors not available:", type(e).__name__, e)

if content is not None:
    with torch.no_grad():
        sims = speaker.similarity(Y, X).cpu().numpy()
        refs, hyps = content.transcribe(X), content.transcribe(Y)
        lp = content.log_probs(X[:1])[0]; lq = content.log_probs(Y[:1])[0]
    kl_t = (lp.exp() * (lp - lq)).sum(-1).cpu().numpy()
    vocab = content.processor.tokenizer.convert_ids_to_tokens(list(range(lp.shape[-1])))

    for i in range(min(3, len(X))):
        print(f"clip {i}  speaker cos = {sims[i]:.3f}   WER = {word_error_rate(refs[i], hyps[i]):.2f}")
        print("   original:", refs[i]); print("   decoded :", hyps[i])
    print(f"mean speaker cosine over {len(X)} clips: {sims.mean():.3f}")

    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True,
                             gridspec_kw={"height_ratios": [3, 3, 1]})
    for ax, l, t in [(axes[0], lp, "original"), (axes[1], lq, "decoded")]:
        ax.imshow(l.exp().cpu().numpy().T, aspect="auto", origin="lower", cmap="Blues", vmin=0, vmax=1)
        ax.set_yticks(range(len(vocab))); ax.set_yticklabels(vocab, fontsize=6)
        ax.set_title(f"character probabilities per frame, {t} (clip 0)")
    axes[2].plot(kl_t); axes[2].set_ylabel("KL"); axes[2].set_xlabel("frame (20 ms)")
    plt.tight_layout(); plt.show()

### 2.8 Granules of the signal itself: the justifiable-granularity envelope

Here the granule is an interval `[a, b]` that summarizes the samples of one
20 ms window (320 samples). For the upper bound `b`, every sample above the
window median `med` is a candidate, and each candidate gets two scores:

* **coverage(b)** = fraction of the window's samples inside `[med, b]`.
  It grows as `b` grows.
* **specificity(b)** = 1 − (b − med)/R, with `R` the signal's range.
  It falls as `b` grows (a wider interval says less).

The chosen `b` maximizes coverage × specificity. The lower bound is chosen the
same way below the median.

**Left:** the three curves for the loudest window of clip 0, with the chosen
`b`. **Middle:** that window's samples, with the granular interval, the peak
interval (min, max) and the 5 %–95 % quantile interval. **Right:** the three
envelopes over the whole clip.

In [ ]:
from gsc.envelope import granular_envelope, peak_envelope, quantile_envelope, robust_range

x = X[0, 0].cpu().numpy().astype(np.float64)
w = 320
R = robust_range(x)
W = x[: len(x) // w * w].reshape(-1, w)
k = np.argmax((W ** 2).mean(1))                    # loudest window
s, med = np.sort(W[k]), np.median(W[k])
cand = s[s >= med]
cov = np.arange(1, len(cand) + 1) / w
spec = np.clip(1 - (cand - med) / R, 0, None)
best = np.argmax(cov * spec)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
axes[0].plot(cand, cov, label="coverage"); axes[0].plot(cand, spec, label="specificity")
axes[0].plot(cand, cov * spec, label="coverage × specificity", lw=2)
axes[0].axvline(cand[best], color="k", ls="--", label=f"chosen b = {cand[best]:.3f}")
axes[0].set_xlabel("candidate upper bound b"); axes[0].legend(fontsize=8)
axes[0].set_title(f"window {k}: choosing b")

lo_g, up_g = granular_envelope(x, w)
lo_q, up_q = quantile_envelope(x, w)
t = np.arange(w) / SR * 1000
axes[1].plot(t, W[k], color="0.6", lw=0.8)
axes[1].axhspan(lo_g[k], up_g[k], color="C0", alpha=0.25, label="granular")
for v, c, lab in [(W[k].max(), "C3", "peak"), (W[k].min(), "C3", None),
                  (up_q[k], "C2", "quantile 5–95 %"), (lo_q[k], "C2", None)]:
    axes[1].axhline(v, color=c, lw=1, label=lab)
inside = np.mean((W[k] >= lo_g[k]) & (W[k] <= up_g[k]))
axes[1].set_title(f"window {k}: granular interval covers {inside:.0%} of samples")
axes[1].set_xlabel("ms"); axes[1].legend(fontsize=8)

tt, tw = np.arange(len(x)) / SR, (np.arange(len(lo_g)) + 0.5) * w / SR
axes[2].plot(tt, x, color="0.8", lw=0.4)
for (lo, up), c, lab in [(peak_envelope(x, w), "C3", "peak"), ((lo_q, up_q), "C2", "quantile"),
                         ((lo_g, up_g), "C0", "granular")]:
    axes[2].plot(tw, up, color=c, lw=1, label=lab); axes[2].plot(tw, lo, color=c, lw=1)
axes[2].set_xlabel("time (s)"); axes[2].legend(fontsize=8); axes[2].set_title("clip 0: envelopes")
plt.tight_layout(); plt.show()

### 2.9 Comparing runs

After training several runs (e.g. `bash sweep.sh ...`, or one run per
quantizer), this plots quality against the measured entropy rate. Compare
quantizers at the same bit rate on the x axis, not at the same codebook size.

In [ ]:
!python plot_results.py --runs runs --out results.png
from IPython.display import Image
Image("results.png")